# 03 · Panel Construction

**Input :** `data/processed/panel_15min_clean.parquet`  
**Outputs:** `data/processed/panel_hourly.parquet` · `data/processed/panel_daily.parquet`

Steps:
0. Load & inspect clean 15-min panel
1. Aggregate to hourly panel + time features
2. Aggregate to daily panel + peak counts + peak ratio + season
3. Final summary

## 0. Imports & Config

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT      = Path().resolve().parent
PROCESSED = ROOT / "data" / "processed"

SEASON_MAP = {
    12: "winter", 1: "winter",  2: "winter",
     3: "spring", 4: "spring",  5: "spring",
     6: "summer", 7: "summer",  8: "summer",
     9: "autumn", 10: "autumn", 11: "autumn",
}

META_COLS = ["low_coverage", "long", "lat", "naam",
             "gemeente", "wegnr", "district", "datum_van"]

# ── load ──────────────────────────────────────────────────────────────────────
df = pd.read_parquet(PROCESSED / "panel_15min_clean.parquet")

# ── structural assertions ─────────────────────────────────────────────────────
EXPECTED_STATIONS = 150
EXPECTED_ROWS     = EXPECTED_STATIONS * 1096 * 96   # 15,782,400
EXPECTED_COLS     = 12
EXPECTED_NAN      = 1_400_593

assert df.shape == (EXPECTED_ROWS, EXPECTED_COLS), \
    f"Shape mismatch: got {df.shape}, expected ({EXPECTED_ROWS}, {EXPECTED_COLS})"
assert df["site ID"].nunique() == EXPECTED_STATIONS, \
    f"Station count mismatch: got {df['site ID'].nunique()}"
assert df["van"].min() == pd.Timestamp("2023-01-01"), \
    f"Start date mismatch: {df['van'].min()}"
assert df["van"].max() == pd.Timestamp("2025-12-31 23:45"), \
    f"End date mismatch: {df['van'].max()}"
assert df["aantal"].dtype == "float32", \
    f"dtype mismatch for aantal: {df['aantal'].dtype}"
assert set(df["van"].dt.hour.unique()) == set(range(24)), \
    "Hour values incomplete — possible timestamp parsing error"

# ── aggregate consistency assertions ─────────────────────────────────────────
nan_count = df["aantal"].isna().sum()
assert nan_count == EXPECTED_NAN, \
    f"NaN count mismatch: got {nan_count:,}, expected {EXPECTED_NAN:,}"

print("✓ Structural and consistency checks passed.")
print(f"\nShape      : {df.shape}")
print(f"\nDtypes:")
print(df.dtypes)
print(f"\nNaN per column:")
print(df.isnull().sum())
print(f"\nDate range : {df['van'].min()} → {df['van'].max()}")
print(f"Stations   : {df['site ID'].nunique()}")
df.head(3)

## 1. Build Hourly Panel

Group four 15-min slots into one hourly slot by flooring the timestamp to the hour, then summing.  
`min_count=1` ensures a result is NaN only when **all four** 15-min values are NaN.

In [9]:
# ── extract site metadata once (constant per station) ─────────────────────────
site_meta = df[["site ID"] + META_COLS].drop_duplicates("site ID")

# ── aggregate 15-min → hourly ─────────────────────────────────────────────────
df["van_h"] = df["van"].dt.floor("h") # floor down to nearest hour for grouping
hourly = (
    df.groupby(["site ID", "van_h"], as_index=False)["aantal"]
    .sum(min_count=1) # if all 4 quarters are NaN, sum should be NaN; otherwise sum the available values
    .rename(columns={"van_h": "van"})
)
df.drop(columns="van_h", inplace=True)   # clean up temp column

# ── time features ─────────────────────────────────────────────────────────────
hourly["year"]             = hourly["van"].dt.year
hourly["month"]            = hourly["van"].dt.month
hourly["day"]              = hourly["van"].dt.day
hourly["hour"]             = hourly["van"].dt.hour
hourly["dayofweek"]        = hourly["van"].dt.dayofweek
hourly["is_weekend"]       = hourly["dayofweek"] >= 5
hourly["is_peak_morning"]  = hourly["hour"].isin([7, 8, 9])
hourly["is_peak_evening"]  = hourly["hour"].isin([16, 17, 18])
hourly["date"]             = hourly["van"].dt.normalize()   # midnight timestamp for weather join

# ── merge metadata ────────────────────────────────────────────────────────────
hourly = hourly.merge(site_meta, on="site ID", how="left")

# ── column order ──────────────────────────────────────────────────────────────
H_COLS = (["site ID", "van", "aantal",
           "year", "month", "day", "hour", "dayofweek",
           "is_weekend", "is_peak_morning", "is_peak_evening", "date"]
          + META_COLS)
hourly = hourly[H_COLS].sort_values(["site ID", "van"]).reset_index(drop=True)

# ── save ──────────────────────────────────────────────────────────────────────
out_h = PROCESSED / "panel_hourly.parquet"
hourly.to_parquet(out_h, index=False)

print(f"Shape          : {hourly.shape}")
print(f"NaN in aantal  : {hourly['aantal'].isna().sum():,}")
print(f"\nSample (5 rows):")
display(hourly[["site ID","van","aantal","hour","dayofweek",
                "is_weekend","is_peak_morning","is_peak_evening","date"]].head(5))

Shape          : (3945600, 20)
NaN in aantal  : 344,463

Sample (5 rows):


,site ID,van,aantal,hour,dayofweek,is_weekend,is_peak_morning,is_peak_evening,date
0,1,2023-01-01 00:00:00,0.0,0,6,True,False,False,2023-01-01
1,1,2023-01-01 01:00:00,1.0,1,6,True,False,False,2023-01-01
2,1,2023-01-01 02:00:00,0.0,2,6,True,False,False,2023-01-01
3,1,2023-01-01 03:00:00,3.0,3,6,True,False,False,2023-01-01
4,1,2023-01-01 04:00:00,0.0,4,6,True,False,False,2023-01-01


## 2. Build Daily Panel

**Steps A–B:** sum total and peak-hour counts per (site, date)  
**Step C:** compute peak ratio = commute share of daily traffic  
**Step D:** add season label  
**Step E:** attach station metadata

In [10]:
# ── Step A: daily total ───────────────────────────────────────────────────────
daily = (
    hourly.groupby(["site ID", "date"], as_index=False)["aantal"]
    .sum(min_count=1)
)

# ── Step B: peak counts ───────────────────────────────────────────────────────
morn = (
    hourly[hourly["is_peak_morning"]]
    .groupby(["site ID", "date"], as_index=False)["aantal"]
    .sum(min_count=1)
    .rename(columns={"aantal": "morning_peak_count"})
)
eve = (
    hourly[hourly["is_peak_evening"]]
    .groupby(["site ID", "date"], as_index=False)["aantal"]
    .sum(min_count=1)
    .rename(columns={"aantal": "evening_peak_count"})
)
daily = daily.merge(morn, on=["site ID", "date"], how="left")
daily = daily.merge(eve,  on=["site ID", "date"], how="left")

# ── Step C: peak_ratio ────────────────────────────────────────────────────────
peak_sum = daily["morning_peak_count"].fillna(0) + daily["evening_peak_count"].fillna(0)
daily["peak_ratio"] = np.where(
    daily["aantal"].isna() | (daily["aantal"] == 0),
    np.nan,
    peak_sum / daily["aantal"]
)

# ── Step D: calendar features ─────────────────────────────────────────────────
daily["year"]      = daily["date"].dt.year
daily["month"]     = daily["date"].dt.month
daily["dayofweek"] = daily["date"].dt.dayofweek
daily["is_weekend"]= daily["dayofweek"] >= 5
daily["season"]    = daily["month"].map(SEASON_MAP)

# ── Step E: metadata ──────────────────────────────────────────────────────────
daily = daily.merge(site_meta, on="site ID", how="left")

# ── column order & save ───────────────────────────────────────────────────────
D_COLS = (["site ID", "date", "aantal",
           "morning_peak_count", "evening_peak_count", "peak_ratio",
           "year", "month", "dayofweek", "is_weekend", "season"]
          + META_COLS)
daily = daily[D_COLS].sort_values(["site ID", "date"]).reset_index(drop=True)

out_d = PROCESSED / "panel_daily.parquet"
daily.to_parquet(out_d, index=False)


print(f"Shape          : {daily.shape}")
print(f"NaN in aantal  : {daily['aantal'].isna().sum():,}")
print(f"\npeak_ratio distribution:")
print(daily["peak_ratio"].describe(percentiles=[.25,.50,.75,.95]).round(3))
print(f"\nSample (5 rows):")
display(daily[["site ID","date","aantal",
               "morning_peak_count","evening_peak_count",
               "peak_ratio","season","naam"]].head(5))

Shape          : (164400, 19)
NaN in aantal  : 14,274

peak_ratio distribution:
count    148281.000
mean          0.423
std           0.128
min           0.000
25%           0.327
50%           0.414
75%           0.514
95%           0.639
max           1.000
Name: peak_ratio, dtype: float64

Sample (5 rows):


,site ID,date,aantal,morning_peak_count,evening_peak_count,peak_ratio,season,naam
0,1,2023-01-01,124.0,5.0,15.0,0.161290,winter,Machelen
1,1,2023-01-02,166.0,32.0,31.0,0.379518,winter,Machelen
2,1,2023-01-03,354.0,70.0,83.0,0.432203,winter,Machelen
3,1,2023-01-04,213.0,43.0,61.0,0.488263,winter,Machelen
4,1,2023-01-05,286.0,70.0,73.0,0.500000,winter,Machelen


## 3. Final Summary

In [12]:
def _nan_pct(s): return f"{s.isna().sum() / len(s) * 100:.1f}%"

rows = [
    ["panel_hourly",
     f"{len(hourly):,}",
     hourly['site ID'].nunique(),
     _nan_pct(hourly['aantal'])],
    ["panel_daily",
     f"{len(daily):,}",
     daily['site ID'].nunique(),
     _nan_pct(daily['aantal'])],
]
hdr = f"{'Panel':<18} {'Rows':>12} {'Stations':>10} {'NaN%':>8}"
sep = "-" * len(hdr)
print(sep)
print(hdr)
print(sep)
for r in rows:
    print(f"{r[0]:<18} {r[1]:>12} {r[2]:>10} {r[3]:>8}")
print(sep)
print(f"\nDate range : {daily['date'].min().date()} → {daily['date'].max().date()}")
print(f"Low-coverage stations flagged : {daily['low_coverage'].any() and daily[daily['low_coverage']]['site ID'].nunique()}")

---------------------------------------------------
Panel                      Rows   Stations     NaN%
---------------------------------------------------
panel_hourly          3,945,600        150     8.7%
panel_daily             164,400        150     8.7%
---------------------------------------------------

Date range : 2023-01-01 → 2025-12-31
Low-coverage stations flagged : 17
